In [13]:
import os
from pathlib import Path
import ants
import numpy as np
from tqdm import tqdm
from bids import BIDSLayout
import nibabel as nib
import json

import glob
import argparse
import multiprocessing
from batchgenerators.utilities.file_and_folder_operations import *
from nnunetv2.paths import nnUNet_raw
from nnunetv2.utilities.dataset_name_id_conversion import find_candidate_datasets
from nnunetv2.dataset_conversion.generate_dataset_json import generate_dataset_json



In [14]:

# extracted traiing.zip file is here
base = '/home/qingyu/datasets/ds004199-1.0.6'
# BIDS_DIR = base
target_dataset_id = 106
target_dataset_name = f'Dataset{target_dataset_id:03.0f}_FCD'
participants = join(base, 'participants.tsv')


OUTPUT_DIR = maybe_mkdir_p(join(nnUNet_raw, target_dataset_name))

imagesTr = join(nnUNet_raw, target_dataset_name, 'imagesTr')
imagesTs = join(nnUNet_raw, target_dataset_name, 'imagesTs')
labelsTr = join(nnUNet_raw, target_dataset_name, 'labelsTr')
maybe_mkdir_p(imagesTr)
maybe_mkdir_p(imagesTs)
maybe_mkdir_p(labelsTr)




In [15]:


# Load BIDS layout
layout = BIDSLayout(base, validate=False)

# Get all subjects with both T1w and FLAIR
subjects = layout.get_subjects()

for sub in tqdm(subjects, desc="Processing subjects"):
    t1w_files = layout.get(subject=sub, suffix="T1w", extension=[".nii", ".nii.gz"], return_type='file')
    flair_files = layout.get(subject=sub, suffix="FLAIR", extension=[".nii", ".nii.gz"], return_type='file')

    if not t1w_files or not flair_files:
        print(f"Skipping {sub} — missing T1w or FLAIR.")
        continue

    t1_file = t1w_files[0]
    flair_file = flair_files[0]

    # Handle non-BIDS ROI manually (e.g., sub-001/anat/sub-001_FLAIR_roi.nii.gz)
    roi_candidates = glob.glob(join(base, f"sub-{sub}", 'anat', '*FLAIR_roi.nii.gz'))
    roi_file = roi_candidates[0] if roi_candidates else None

    # Read and register images
    t1_img = ants.image_read(str(t1_file))
    flair_img = ants.image_read(str(flair_file))

    t1_img_nib = nib.load(str(t1_file))

    
    tx = ants.registration(fixed=t1_img, moving=flair_img, type_of_transform="Affine")
    flair_reg = tx["warpedmovout"]

    # Save nnU-Net modalities
    out_base = f"sub-{sub}"
    # nib.save(nib.Nifti1Image(t1_img.numpy(), t1_img.affine), imagesTr / f"{out_base}_0000.nii.gz")
    # nib.save(nib.Nifti1Image(flair_reg.numpy(), t1_img.affine), imagesTr / f"{out_base}_0001.nii.gz")
    nib.save(nib.Nifti1Image(t1_img.numpy(), t1_img_nib.affine), join(imagesTr , f"{out_base}_0000.nii.gz"))
    nib.save(nib.Nifti1Image(flair_reg.numpy(), t1_img_nib.affine), join(imagesTr , f"{out_base}_0001.nii.gz"))

    
    # Process ROI
    if roi_file:
        roi_img = ants.image_read(str(roi_file))
        roi_reg = ants.apply_transforms(fixed=t1_img, moving=roi_img,
                                        transformlist=tx['fwdtransforms'],
                                        interpolator='nearestNeighbor')
        roi_data = roi_reg.numpy().astype(np.uint8)
    else:
        roi_data = np.zeros(t1_img.shape, dtype=np.uint8)

    nib.save(nib.Nifti1Image(roi_data, t1_img_nib.affine), join(labelsTr , f"{out_base}.nii.gz"))

print("✅ Done. nnU-Net format data is ready.")

Processing subjects: 100%|█████████████████████████████████████████| 170/170 [38:24<00:00, 13.56s/it]

✅ Done. nnU-Net format data is ready.
